# NMR matching task (work copy)
Match molecules A/B/C to spectra via Trotterized NMR simulation.

In [1]:
from struqture_py.spins import PauliHamiltonian
from qiskit.quantum_info.operators import SparsePauliOp
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter

# NEW imports for Qiskit 2.2.x
from qiskit.primitives import StatevectorEstimator as Estimator

import matplotlib.pyplot as plt
from itertools import product
import json
import numpy as np
from pathlib import Path
import pickle
from typing import Tuple, Any
import re

## 1. Load Hamiltonian

In [2]:
# Folder that contains the Hamiltonian JSON files.
data_dir = Path("files")

# Name of the molecule.
molecule_name = "molecule A"  

# Magnetic field
field_T = 11.7433

# Build full filename.
filename = data_dir / f"{molecule_name}_B{field_T}T.json"

# Open JSON file.
with open(filename, "r") as json_file:
    hamiltonian_json = json.load(json_file)

# Create PauliHamiltonian from JSON file.
hamiltonian_struqture = PauliHamiltonian.from_json(hamiltonian_json)

# Get the number of spins in the Hamiltonian.
number_spins = hamiltonian_struqture.current_number_spins()

# Note that the energies are orders of magnitude larger than the couplings.
print("Struqture Hamiltonian:\n", hamiltonian_struqture)

Struqture Hamiltonian:
 PauliHamiltonian{
0Z: -1.5708053977817934e9,
1Z: -1.570805680525128e9,
2Z: -1.5708049108349397e9,
0X1X: 1.413716694115407e0,
0Y1Y: 1.413716694115407e0,
0Z1Z: 1.413716694115407e0,
0X2X: 1.853539665617978e1,
0Y2Y: 1.853539665617978e1,
0Z2Z: 1.853539665617978e1,
1X2X: 2.8117254249628648e1,
1Y2Y: 2.8117254249628648e1,
1Z2Z: 2.8117254249628648e1,
}


## 2. Rotating frame (subtract mean Z)

In [3]:
# We are now extracting the single-qubit Z terms from the full Hamiltonian:
z_terms = []
for i in range(number_spins):
    z_terms.append(hamiltonian_struqture.get(f"{i}Z"))

# Mean of Z terms.
mean_z_parameter = np.mean(z_terms)

# Setting the chemical shift due to rotating frame of the QC calculation:
chemical_shift_hz = mean_z_parameter

# We now subtract the mean value of the Z coupling from the full Hamiltonian
# to arrive at the Hamiltonian that is used in the NMR calculation and store the second
# chemical shift we need to apply to the spectrum at the end.
correction_hamiltonian = PauliHamiltonian()
for i in range(number_spins):
    correction_hamiltonian.set(f"{i}Z", mean_z_parameter)

hamiltonian_struqture = hamiltonian_struqture - correction_hamiltonian

# In the rotating frame, the energies are all of order 1.
print("Hamiltonian in rotating frame: \n", hamiltonian_struqture)

Hamiltonian in rotating frame: 
 PauliHamiltonian{
0Z: -6.806783962249756e1,
1Z: -3.508111741542816e2,
2Z: 4.1887901401519775e2,
0X1X: 1.413716694115407e0,
0Y1Y: 1.413716694115407e0,
0Z1Z: 1.413716694115407e0,
0X2X: 1.853539665617978e1,
0Y2Y: 1.853539665617978e1,
0Z2Z: 1.853539665617978e1,
1X2X: 2.8117254249628648e1,
1Y2Y: 2.8117254249628648e1,
1Z2Z: 2.8117254249628648e1,
}


## 3. Struqture -> Qiskit

In [4]:
def struqture_hamiltonian_to_qiskit_op(
    ham_struqture: PauliHamiltonian,
    n_qubits: int,
    reverse_qubit_order: bool = True,
) -> SparsePauliOp:
    """
    Convert struqture PauliHamiltonian keys like '0Z', '10X11X' into Qiskit labels.

    Args:
        ham_struqture (PauliHamiltonian): struqture Hamiltonian to transform to Qiskit
        n_qubits (int): number of qubits in the Hamiltonian
        reverse_qubit_order (bool): makes Qiskit's rightmost char qubit 0 (little-endian).
    """
    labels = []
    coeffs = []
    token_re = re.compile(r"(\d+)([XYZ])")

    for key, val in zip(ham_struqture.keys(), ham_struqture.values()):
        s = str(key)  # e.g., '0Z', '0X1X', '10X11X'
        pauli = ["I"] * n_qubits
        if s != "I":
            for m in token_re.finditer(s):
                idx = int(m.group(1))  # site index (can be multi-digit)
                op = m.group(2)  # 'X', 'Y', or 'Z'
                q = (n_qubits - 1 - idx) if reverse_qubit_order else idx
                if not (0 <= q < n_qubits):
                    raise IndexError(
                        f"Site index {idx} (mapped to qubit {q}) out of range 0..{n_qubits-1}"
                    )
                pauli[q] = op
        labels.append("".join(pauli))
        coeffs.append(complex(val))

    return SparsePauliOp(labels, coeffs)

In [5]:
hamiltonian_qiskit = struqture_hamiltonian_to_qiskit_op(hamiltonian_struqture, number_spins)

# Qiskit representation of the Hamiltonian.
print("Hamiltonian in Qiskit:\n", hamiltonian_qiskit)

Hamiltonian in Qiskit:
 SparsePauliOp(['IIZ', 'IZI', 'ZII', 'IXX', 'IYY', 'IZZ', 'XIX', 'YIY', 'ZIZ', 'XXI', 'YYI', 'ZZI'],
              coeffs=[ -68.06783962+0.j, -350.81117415+0.j,  418.87901402+0.j,
    1.41371669+0.j,    1.41371669+0.j,    1.41371669+0.j,
   18.53539666+0.j,   18.53539666+0.j,   18.53539666+0.j,
   28.11725425+0.j,   28.11725425+0.j,   28.11725425+0.j])


## 4. Trotter circuit & initial states

### 4.1 Trotter circuit

In [6]:
# Default number of Trottersteps.
number_of_trottersteps = 300

# Default Trotter timestep.
trotter_timestep = 0.0025



# Trotter Circuit
trotter_step_first_order = PauliEvolutionGate(
    hamiltonian_qiskit, trotter_timestep, synthesis=LieTrotter()
)

circuit = QuantumCircuit(hamiltonian_qiskit.num_qubits)
circuit.append(trotter_step_first_order, range(hamiltonian_qiskit.num_qubits))
circuit = circuit.decompose(reps=1)

print("Trotter circuit: \n", circuit)

Trotter circuit: 
      ┌──────────────┐┌─────────────────┐┌─────────────────┐                »
q_0: ┤ Rz(-0.34034) ├┤0                ├┤0                ├─■──────────────»
     ├─────────────┬┘│  Rxx(0.0070686) ││  Ryy(0.0070686) │ │ZZ(0.0070686) »
q_1: ┤ Rz(-1.7541) ├─┤1                ├┤1                ├─■──────────────»
     └┬────────────┤ └─────────────────┘└─────────────────┘                »
q_2: ─┤ Rz(2.0944) ├───────────────────────────────────────────────────────»
      └────────────┘                                                       »
«     ┌────────────────┐┌────────────────┐                                »
«q_0: ┤0               ├┤0               ├─■──────────────────────────────»
«     │                ││                │ │             ┌───────────────┐»
«q_1: ┤  Rxx(0.092677) ├┤  Ryy(0.092677) ├─┼─────────────┤0              ├»
«     │                ││                │ │ZZ(0.092677) │  Rxx(0.14059) │»
«q_2: ┤1               ├┤1               ├─■─────────────┤1   

### 4.2 Initial states (X-basis product states)

Each qubit is prepared as $RZ(\phi)\,H\,|0\rangle \propto (|0\rangle + e^{i\phi}|1\rangle)/\sqrt{2}$, so $\phi=0$ gives $|+\rangle$ and $\phi=\pi$ gives $|-\rangle$. Enumerating all $2^N$ bit strings $b$ with $\phi_i = \pi b_i$ gives the complete eigenbasis of $\sum_i X_i$, with eigenvalue $\sum_i (-1)^{b_i}$.

In [ ]:
from qiskit.circuit import ParameterVector


def xsum_eigenstate_template(n_qubits: int):
    """Parameterized prep for any eigenstate of sum_i X_i.

    Per qubit:  |0> --H--> |+> --RZ(phi)--> (|0> + e^{i phi}|1>)/sqrt(2)   (up to a global phase)
        phi = 0   ->  |+>   (X eigenvalue +1)
        phi = pi  ->  |->   (X eigenvalue -1)
    """
    phi = ParameterVector("phi", n_qubits)
    qc = QuantumCircuit(n_qubits, name="XsumEigen")

    for q in range(n_qubits):
        qc.h(q)  # |0> -> |+>
        qc.rz(phi[q], q)  # phi = pi flips |+> -> |->

    return qc, phi


# Create the parametrized circuit.
qc_init_template, phi = xsum_eigenstate_template(number_spins)

# --- Enumerate initial states and build parameter maps ---
# ALL 2^N X-basis product states. Bit b_i = 0 means |+> on qubit i, b_i = 1 means |->.
bits_list = list(product((0, 1), repeat=number_spins))

# One parameter binding per initial state: phi[i] = pi * b_i.
param_maps = [{phi[i]: np.pi * b for i, b in enumerate(bits)} for bits in bits_list]

print(f"{len(bits_list)} initial states for {number_spins} spins")
print(qc_init_template)

8 initial states for 3 spins
     ┌───┐┌────────────┐
q_0: ┤ H ├┤ Rz(phi[0]) ├
     ├───┤├────────────┤
q_1: ┤ H ├┤ Rz(phi[1]) ├
     ├───┤├────────────┤
q_2: ┤ H ├┤ Rz(phi[2]) ├
     └───┘└────────────┘


In [8]:
# Sanity check of the initial states (cheap: 2^N statevectors, no time evolution).
from qiskit.quantum_info import Statevector

x_tot_matrix = SparsePauliOp.from_sparse_list(
    [("X", [i], 1.0) for i in range(number_spins)], num_qubits=number_spins
).to_matrix()

states = []
for bits, pmap in zip(bits_list, param_maps):
    psi = Statevector(qc_init_template.assign_parameters(pmap)).data
    m_x = sum((-1) ** b for b in bits)  # expected eigenvalue of sum_i X_i
    assert np.allclose(x_tot_matrix @ psi, m_x * psi), f"{bits} is not an eigenstate"
    states.append(psi)
    print(f"bits {bits}:  sum_i X_i eigenvalue = {m_x:+d}")

# The 2^N states must form a complete orthonormal basis (resolution of the identity in the trace).
gram = np.array(states).conj() @ np.array(states).T
assert np.allclose(gram, np.eye(2**number_spins)), "states are not an orthonormal basis"
print("All states are eigenstates of sum_i X_i and form a complete orthonormal basis.")

bits (0, 0, 0):  sum_i X_i eigenvalue = +3
bits (0, 0, 1):  sum_i X_i eigenvalue = +1
bits (0, 1, 0):  sum_i X_i eigenvalue = +1
bits (0, 1, 1):  sum_i X_i eigenvalue = -1
bits (1, 0, 0):  sum_i X_i eigenvalue = +1
bits (1, 0, 1):  sum_i X_i eigenvalue = -1
bits (1, 1, 0):  sum_i X_i eigenvalue = -1
bits (1, 1, 1):  sum_i X_i eigenvalue = -3
All states are eigenstates of sum_i X_i and form a complete orthonormal basis.


## 5. Simulation (Estimator)

### 5.1 Observables

In [10]:
obs_x = SparsePauliOp.from_sparse_list(
    [("X", [i], 1.0) for i in range(number_spins)], num_qubits=number_spins
)
obs_y = SparsePauliOp.from_sparse_list(
    [("Y", [i], 1.0) for i in range(number_spins)], num_qubits=number_spins
)

### 5.2 Run & collect expectation values

In [11]:
# Prebuild the parametric circuit at each step: U^n composed after the init template.
circuits_per_step = []
c = QuantumCircuit(number_spins)
c = c.compose(qc_init_template)  # still parametric in phi
circuits_per_step.append(c)

for _ in range(1, number_of_trottersteps):
    c = c.compose(circuit)  # append one more Trotter step
    circuits_per_step.append(c)

number_of_initializations = len(bits_list)
exp_x = np.zeros((number_of_initializations, number_of_trottersteps))
exp_y = np.zeros((number_of_initializations, number_of_trottersteps))

estimator = Estimator()

# Batch all initial states for each step to minimize overhead
for k, circ in enumerate(circuits_per_step):
    pubs = []
    for m in range(number_of_initializations):
        pubs.append((circ, [obs_x, obs_y], param_maps[m]))  # pass parameter_values per pub
    job = estimator.run(pubs)
    results = job.result()  # list-like; one result per pub
    for m, res in enumerate(results):
        evs = res.data.evs  # [⟨obs_x⟩, ⟨obs_y⟩]
        exp_x[m, k] = evs[0]
        exp_y[m, k] = evs[1]

### 5.3 Correlation functions C_xx, C_yx

In [ ]:
# Expectation values.
c_xx = np.zeros(number_of_trottersteps)
c_yx = np.zeros(number_of_trottersteps)

# Perform vectorized calculations
mx_0 = exp_x[:, 0]
for i in range(number_of_trottersteps):
    c_xx[i] = np.sum(mx_0 * exp_x[:, i], axis=0)
    c_yx[i] = np.sum(mx_0 * exp_y[:, i], axis=0)

# --- apply a single exponential window before FFT ---
noise_strength = 5.0  # s^-1 (choose per system/step size)
times = np.arange(number_of_trottersteps) * trotter_timestep
window = np.exp(-noise_strength * times)
c_xx = c_xx * window
c_yx = c_yx * window

In [ ]:
plt.figure(dpi=100)
plt.plot(times, c_xx, color="C0", lw=2.0)
plt.xlabel("Time", fontsize=10)
plt.ylabel("Magnetization in x direction", fontsize=10)
plt.show()

## 6. Spectrum (FFT)

In [ ]:
def time_signal_to_spectrum(
    c_xx: np.ndarray,
    c_yx: np.ndarray,
    trotter_timestep: float,
    number_of_trottersteps: int,
    Hz_to_ppm_conversion: float,
    chem_shift_hz: float,
) -> Tuple[np.ndarray[Any, Any], np.ndarray[Any, Any]]:
    """Calculate the full spectrum from the computed expectation values via a Fourier Transform.

    Args:
        m_avg_z (np.ndarray): The first expectation value (c_xx)
        m_avg_y (np.ndarray): The second expectation value (c_yx)
        N_max (int): Number of Trotter steps.
        trotter_timestep (float): The Trotter timestep.
        Hz_to_ppm_conversion (float): Conversion factor from frequency to ppm.
        chem_shift_hz (float): The chemical shift of the input Hamiltonian (in Hz).

    Returns:
        Tuple[np.ndarray, np.ndarray]: A tuple containing:
            - **spectrum_full (np.ndarray):** The calculated spectrum.
            - **ppm (np.ndarray):** The ppm array to plot the full spectrum against.
    """
    spectrum_full = np.zeros(c_xx.shape[0], dtype=np.float64)

    m_avg_z = fft_causal(c_xx)
    m_avg_y = fft_causal(c_yx)

    frequencies = np.fft.fftshift(np.fft.fftfreq(number_of_trottersteps, d=trotter_timestep))

    # And now we have the frequency axis in ppm:
    # we need to shift the frequencies by twice the chemical input shift and then by the reference
    # energy, i.e. the energy of the honomuclear isotope (H) times the magnetic field.
    
    # the chemical shift:
    chem_shift = 2 * chem_shift_hz / (2.0*np.pi/Hz_to_ppm_conversion) + 1e6

    # to have the frequency axis in ppm:
    ppm = (frequencies) * Hz_to_ppm_conversion - chem_shift

    spectrum_full = m_avg_z.real + m_avg_y.imag

    spectrum_full /= np.max(np.abs(spectrum_full))

    return np.abs(spectrum_full), ppm


def fft_causal(f: np.ndarray) -> np.ndarray:
    """
    Performs the Fourier Transform of a causal signal by separating its even and odd parts.

    Args:
        f (np.ndarray): Input causal signal.

    Returns:
        np.ndarray: The complex Fourier Transform of the causal signal.
    """
    L = len(f)

    # Even part (symmetrize)
    fe = np.zeros(2 * L - 1, dtype=f.dtype)
    fe[:L] = f
    fe[L:] = f[:-1][::-1]
    fe /= 2

    # Odd part (antisymmetrize)
    fo = np.zeros(2 * L - 1, dtype=f.dtype)
    fo[:L] = f
    fo[L:] = -f[:-1][::-1]
    fo /= 2
    fo[0] = 0  # Avoids jump discontinuity at t=0

    # FFT with orthogonal normalization and shift
    fft_even = np.fft.fftshift(np.fft.fft(fe, norm="ortho"))
    fft_odd = np.fft.fftshift(np.fft.fft(fo, norm="ortho"))

    # Combine and downsample to original length
    combined_fft = (fft_even + fft_odd)[::2]

    return combined_fft

In [ ]:
# Gyromagnetic ratio for 1H.
GYROMAGNETIC_RATIO_1H_RAD_S_T = 267_522_128.0

# Larmor/reference frequency in Hz at the given field.
reference_energy_hz = GYROMAGNETIC_RATIO_1H_RAD_S_T * field_T / (2 * np.pi)

# Conversion factor to go from Hz to ppm at this field: ppm = (Hz) * (1e6 / v0)
Hz_to_ppm_conversion = 1e6 / reference_energy_hz

In [ ]:
spectrum_full, ppm = time_signal_to_spectrum(
    c_xx,
    c_yx,
    trotter_timestep,
    number_of_trottersteps,
    Hz_to_ppm_conversion=Hz_to_ppm_conversion,
    chem_shift_hz=chemical_shift_hz,
)

### 6.1 Reference spectrum

In [ ]:
with open(data_dir / f"spectrum_3.pkl", "rb") as f:
    hqs_spectrum_tools_results = pickle.load(f)

hqs_spectrum_tools_ppm = hqs_spectrum_tools_results["frequencies"]
hqs_spectrum_tools_spectrum = hqs_spectrum_tools_results["spectrum"]
hqs_spectrum_tools_spectrum /= np.max(np.abs(hqs_spectrum_tools_spectrum))

## 7. Matching task

In [ ]:
plt.figure(dpi=100)
plt.plot(
    hqs_spectrum_tools_ppm, hqs_spectrum_tools_spectrum, color="C0", label="HQS Spectrum Tools"
)
plt.plot(ppm, spectrum_full, color="C1", ls="--", label="Simulation on Quantum Computer")

plt.xlabel("Frequency [ppm]")
plt.ylabel("Spectrum")
plt.legend(fontsize=7)
plt.show()

## 8. Noise tolerance sweep

`noise_strength` is the decay rate $\Gamma$ of the window $e^{-\Gamma t}$ applied to the time signal. It turns every line into a Lorentzian of full width $\Gamma/\pi$ (in Hz). The first cell below sweeps it from 0.1 to 100 s$^{-1}$ for the molecule set in section 1, reusing the expectation values of section 5.2, and counts the resolved peaks at each value. A peak counts as resolved if it rises at least 5% of the tallest peak above the valley next to it. The maximum tolerable `noise_strength` is the largest value at which no peak has been lost. The second cell draws the result. Both save into the folder `noise sweep`.

Results for the three molecules (run once per molecule by changing `molecule_name` in section 1):

| Molecule | J couplings [Hz] | Smallest line spacing we resolve [Hz] | Maximum tolerable `noise_strength` [s$^{-1}$] | Line width there, $\Gamma/\pi$ [Hz] |
|---|---|---|---|---|
| A | 0.9, 11.8, 17.9 | 6.1 | 20.5 | 6.5 |
| B | 0.5, 8.6, 15.6 | 7.0 | 23.5 | 7.5 |
| C | 7.9 | 7.9 | 24.5 | 7.8 |

**Relation to the Hamiltonian.** In all three molecules the peaks stop being resolved when the line width $\Gamma/\pi$ reaches the smallest spacing between neighbouring lines, so $\Gamma_{\max} \approx \pi\,\Delta\nu_{\min}$. That spacing is set by the J couplings: in C it is J itself (7.9 Hz); in A and B it is the inner gap of a doublet of doublets, the difference of two couplings (17.9 − 11.8 = 6.1 Hz and 15.6 − 8.6 = 7.0 Hz).

**Limit of the default settings.** The signal lasts 0.75 s, which gives a frequency grid of 1.33 Hz. The 0.9 Hz coupling of A and the 0.5 Hz coupling of B are below that and are not resolved at any `noise_strength` (A shows 8 of the 12 reference lines). The exact maximum also shifts a little with the 5% criterion.

In [ ]:
from scipy.signal import find_peaks

# Reference file of each molecule (result of the matching task) and the folder for the results.
matched_spectrum = {"molecule A": "spectrum_3", "molecule B": "spectrum_2", "molecule C": "spectrum_1"}
sweep_dir = Path("noise sweep")
sweep_dir.mkdir(exist_ok=True)

# Correlation functions of section 5.2 without a window, so the simulation is not repeated.
raw_c_xx = (exp_x[:, 0][:, None] * exp_x).sum(axis=0)
raw_c_yx = (exp_x[:, 0][:, None] * exp_y).sum(axis=0)


def spectrum_at(noise):
    """Spectrum of the current molecule for one value of noise_strength."""
    decay = np.exp(-noise * times)
    spec, axis = time_signal_to_spectrum(
        raw_c_xx * decay,
        raw_c_yx * decay,
        trotter_timestep,
        number_of_trottersteps,
        Hz_to_ppm_conversion=Hz_to_ppm_conversion,
        chem_shift_hz=chemical_shift_hz,
    )
    return np.asarray(axis, dtype=float), np.asarray(spec, dtype=float)


def resolved_peaks(spec):
    """A peak is resolved if it rises at least 5% of the tallest peak above the valley next to it."""
    return len(find_peaks(spec, prominence=0.05)[0])


# The sweep: 0.1 s^-1, then 0.5 to 100 s^-1 in steps of 0.5.
noise_values = np.concatenate([[0.1], np.arange(0.5, 100.5, 0.5)])
peak_counts = np.array([resolved_peaks(spectrum_at(noise)[1]) for noise in noise_values])
all_peaks = peak_counts.max()
max_tolerable_noise = noise_values[peak_counts == all_peaks].max()

# Reference: line positions, and the smallest line spacing that our frequency grid can show.
with open(data_dir / f"{matched_spectrum[molecule_name]}.pkl", "rb") as pkl_file:
    reference = pickle.load(pkl_file)
reference_ppm = np.asarray(reference["frequencies"], dtype=float)
reference_spectrum = np.asarray(reference["spectrum"], dtype=float)
reference_spectrum = reference_spectrum / reference_spectrum.max()
reference_lines_ppm = np.sort(reference_ppm[find_peaks(reference_spectrum, prominence=0.05)[0]])
line_spacings_hz = np.diff(reference_lines_ppm) / Hz_to_ppm_conversion
grid_resolution_hz = 1 / (number_of_trottersteps * trotter_timestep)
closest_pair = np.where(line_spacings_hz > grid_resolution_hz, line_spacings_hz, np.inf).argmin()
smallest_spacing_hz = line_spacings_hz[closest_pair]

# J couplings from the Hamiltonian: the XX coefficient is 2*pi*J / 4.
j_couplings_hz = sorted(
    abs(2 * coeff.real / np.pi)
    for label, coeff in zip(hamiltonian_qiskit.paulis.to_labels(), hamiltonian_qiskit.coeffs)
    if label.count("X") == 2
)

print(f"{molecule_name} against {matched_spectrum[molecule_name]}")
print(f"  J couplings [Hz]: {', '.join(f'{j:.1f}' for j in j_couplings_hz)}")
print(f"  reference lines: {len(reference_lines_ppm)}, resolved by our {grid_resolution_hz:.2f} Hz grid: {all_peaks}")
print(f"  smallest line spacing we can resolve: {smallest_spacing_hz:.1f} Hz")
print(f"  maximum tolerable noise_strength: {max_tolerable_noise:g} s^-1")
print(f"  line width there (noise_strength / pi): {max_tolerable_noise / np.pi:.1f} Hz")

# Save the sweep and add this molecule to the summary of all molecules.
np.savetxt(
    sweep_dir / f"{molecule_name} sweep.csv",
    np.column_stack([noise_values, peak_counts]),
    delimiter=",",
    header="noise_strength_per_s,resolved_peaks",
    comments="",
    fmt=["%.1f", "%d"],
)
summary_file = sweep_dir / "summary.json"
summary = json.loads(summary_file.read_text()) if summary_file.exists() else {}
summary[molecule_name] = {
    "reference": matched_spectrum[molecule_name],
    "j_couplings_hz": [round(j, 2) for j in j_couplings_hz],
    "resolved_peaks": int(all_peaks),
    "smallest_resolvable_line_spacing_hz": round(float(smallest_spacing_hz), 2),
    "max_tolerable_noise_strength_per_s": float(max_tolerable_noise),
    "line_width_at_maximum_hz": round(float(max_tolerable_noise / np.pi), 2),
}
summary_file.write_text(json.dumps(summary, indent=2, sort_keys=True))

In [ ]:
# Same plot as in the matching task (full spectrum, same colours) at six points of the sweep,
# from its start at 0.1 s^-1 up to four times the maximum tolerable value.
shown = {
    0.1: "start of the sweep",
    1.0: "",
    5.0: "notebook default",
    max_tolerable_noise: "maximum tolerable",
    2 * max_tolerable_noise: "2 x maximum",
    4 * max_tolerable_noise: "4 x maximum",
}

fig, axes = plt.subplots(2, 3, figsize=(18, 9), dpi=100)
for ax, (noise, note) in zip(axes.ravel(), shown.items()):
    axis, spec = spectrum_at(noise)
    ax.plot(reference_ppm, reference_spectrum, color="C0", label="HQS Spectrum Tools")
    ax.plot(axis, spec, color="C1", ls="--", label="Simulation on Quantum Computer")
    note = f" ({note})" if note else ""
    ax.set_title(f"noise_strength = {noise:g} s$^{{-1}}${note}: {resolved_peaks(spec)} peaks resolved", fontsize=10)
    ax.set_xlabel("Frequency [ppm]")
    ax.set_ylabel("Spectrum")
    ax.legend(fontsize=7)
fig.suptitle(f"{molecule_name} against {matched_spectrum[molecule_name]}", fontsize=12)
fig.tight_layout()
fig.savefig(sweep_dir / f"{molecule_name} sweep.png", dpi=200, bbox_inches="tight")
plt.show()